# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rohmasaeed/flyrank-ml-internship-/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from pathlib import Path

DATA_PATH = Path("/content/content_refresh_anonymized (1).csv")

print("Dataset path:", DATA_PATH)
print("File exists:", DATA_PATH.exists())

Dataset path: /content/content_refresh_anonymized (1).csv
File exists: True


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [3]:
print(df.columns.tolist())
required = [
    "content_id",
    "days_since_last_update",
    "ctr",
    "avg_position",
]

missing = [c for c in required if c not in df.columns]

print("Missing columns:", missing)

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Missing columns: []


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. My rule and its reason codes

### Rule

I will prioritize content for refresh when it shows evidence of either content staleness or weak CTR relative to its current ranking position.

The baseline uses two signals:

1. `days_since_last_update` — content staleness, linked to the refresh/staleness logic.
2. `ctr` together with `avg_position` — low CTR at a useful ranking position, linked to CTR-fix logic.

The final score will combine points from these signals. Higher scores indicate higher refresh priority. This is a directional, decision-support baseline, not a causal model.

### Reason codes

- `STALE_REFRESH` — the content is old enough to receive staleness points.
- `LOW_CTR_POSITION` — CTR is low relative to a useful ranking position.
- `STALE_PLUS_LOW_CTR` — both signals contribute to the score.
- `NO_PRIORITY_SIGNAL` — neither condition contributes points.

### Action labels

- `REFRESH` — positive baseline score.
- `MONITOR` — no positive baseline signal.

The thresholds will be checked against the observed dataset before the final queue is interpreted.

In [5]:
# ==========================================
# SECTION 1 — SIGNAL AUDIT
# ==========================================

import numpy as np
import pandas as pd

# Make a working copy
audit = df.copy()

# ------------------------------------------
# SIGNAL 1: DAYS SINCE LAST UPDATE
# ------------------------------------------

stale_bins = [-1, 90, 180, 365, np.inf]
stale_labels = ["0–90", "91–180", "181–365", "365+"]

audit["stale_bucket"] = pd.cut(
    audit["days_since_last_update"],
    bins=stale_bins,
    labels=stale_labels,
    include_lowest=True
)

stale_table = (
    audit.groupby("stale_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        median_engagement=("engagement_rate", "median"),
        median_position=("avg_position", "median")
    )
    .reset_index()
)

print("SIGNAL 1 — STALENESS")
display(stale_table)

print("\nVerdict: MIXED")
print(
    "Reason: the observed performance pattern is not perfectly monotonic "
    "across age buckets, so staleness is useful as a directional "
    "decision-support signal but should not be treated as causal."
)


# ------------------------------------------
# SIGNAL 2: CTR VS POSITION
# ------------------------------------------

position_audit = audit[audit["avg_position"] > 0].copy()

position_bins = [0, 3, 10, 20, np.inf]
position_labels = ["1–3", "4–10", "11–20", "21+"]

position_audit["position_bucket"] = pd.cut(
    position_audit["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

position_audit["low_ctr"] = position_audit["ctr"] < 0.05

ctr_position_table = (
    position_audit.groupby("position_bucket", observed=False)
    .agg(
        n=("content_id", "size"),
        median_ctr=("ctr", "median"),
        low_ctr_rate=("low_ctr", "mean")
    )
    .reset_index()
)

print("\nSIGNAL 2 — CTR VS POSITION")
display(ctr_position_table)

print("\nVerdict: MIXED")
print(
    "Reason: CTR varies by ranking position, but the relationship is not "
    "perfectly monotonic. Therefore low CTR at a useful ranking position "
    "is treated as a directional signal rather than proof of a content problem."
)

SIGNAL 1 — STALENESS


,stale_bucket,n,median_ctr,median_engagement,median_position
0,0–90,20655,0.04,0.0,10.0
1,91–180,9171,0.10,0.0,13.6
2,181–365,169,0.00,0.0,7.0
3,365+,5,0.00,0.0,7.5



Verdict: MIXED
Reason: the observed performance pattern is not perfectly monotonic across age buckets, so staleness is useful as a directional decision-support signal but should not be treated as causal.

SIGNAL 2 — CTR VS POSITION


,position_bucket,n,median_ctr,low_ctr_rate
0,1–3,1141,0.00,0.535495
1,4–10,11842,0.16,0.360919
2,11–20,7273,0.10,0.413172
3,21+,8539,0.00,0.585432



Verdict: MIXED
Reason: CTR varies by ranking position, but the relationship is not perfectly monotonic. Therefore low CTR at a useful ranking position is treated as a directional signal rather than proof of a content problem.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Build the ranked queue

The baseline score is intentionally simple and transparent.

### Staleness points

- 0–90 days → 0
- 91–180 days → 1
- 181–365 days → 2
- 365+ days → 3

### CTR/position points

- Position 4–10 and CTR < 0.05 → 3
- Position 4–10 and CTR 0.05–<0.10 → 2
- Position 11–20 and CTR < 0.05 → 1
- Otherwise → 0

The final action score is:

`action_score = staleness_points + ctr_position_points`

Positive scores receive `REFRESH`; zero receives `MONITOR`.

The queue is ranked from highest to lowest score.

In [7]:
# ==========================================
# SECTION 2 — BUILD RANKED QUEUE
# ==========================================

queue = df.copy()

# ------------------------------------------
# STALENESS SCORE
# ------------------------------------------

queue["staleness_points"] = np.select(
    [
        queue["days_since_last_update"].between(0, 90),
        queue["days_since_last_update"].between(91, 180),
        queue["days_since_last_update"].between(181, 365),
        queue["days_since_last_update"] > 365
    ],
    [
        0,
        1,
        2,
        3
    ],
    default=0
)

# ------------------------------------------
# CTR + POSITION SCORE
# ------------------------------------------

queue["ctr_position_points"] = np.select(
    [
        (
            queue["avg_position"].between(4, 10)
            & (queue["ctr"] < 0.05)
        ),

        (
            queue["avg_position"].between(4, 10)
            & queue["ctr"].between(0.05, 0.10, inclusive="left")
        ),

        (
            queue["avg_position"].between(11, 20)
            & (queue["ctr"] < 0.05)
        )
    ],
    [
        3,
        2,
        1
    ],
    default=0
)

# ------------------------------------------
# FINAL SCORE
# ------------------------------------------

queue["action_score"] = (
    queue["staleness_points"]
    + queue["ctr_position_points"]
).astype(int)

# ------------------------------------------
# ONE REASON CODE PER ROW
# ------------------------------------------

queue["reason_code"] = np.select(
    [
        (
            (queue["staleness_points"] > 0)
            & (queue["ctr_position_points"] > 0)
        ),

        queue["staleness_points"] > 0,

        queue["ctr_position_points"] > 0
    ],
    [
        "STALE_PLUS_LOW_CTR",
        "STALE_REFRESH",
        "LOW_CTR_POSITION"
    ],
    default="NO_PRIORITY_SIGNAL"
)

# ------------------------------------------
# ACTION
# ------------------------------------------

queue["action"] = np.where(
    queue["action_score"] > 0,
    "REFRESH",
    "MONITOR"
)

# ------------------------------------------
# RANK
# ------------------------------------------

queue = queue.sort_values(
    [
        "action_score",
        "staleness_points",
        "ctr_position_points",
        "content_id"
    ],
    ascending=[
        False,
        False,
        False,
        True
    ]
).reset_index(drop=True)

queue["rank"] = np.arange(1, len(queue) + 1)

# ------------------------------------------
# OUTPUT COLUMNS
# ------------------------------------------

output_columns = [
    "rank",
    "content_id",
    "action_score",
    "reason_code",
    "action",
    "days_since_last_update",
    "ctr",
    "avg_position",
    "staleness_points",
    "ctr_position_points"
]

output = queue[output_columns].copy()

# ------------------------------------------
# WRITE REQUIRED CSV
# ------------------------------------------

output_path = Path("work/outputs/baseline_action_score.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

output.to_csv(
    output_path,
    index=False
)

print("CSV successfully written:")
print(output_path)

print("\nRows:", len(output))

print("\nTop 10:")
display(output.head(10))
print("Score distribution:")
print(output["action_score"].value_counts().sort_index())

print("\nReason-code distribution:")
print(output["reason_code"].value_counts())

print("\nAction distribution:")
print(output["action"].value_counts())
display(output.head(20))

CSV successfully written:
work/outputs/baseline_action_score.csv

Rows: 30000

Top 10:


,rank,content_id,action_score,reason_code,action,days_since_last_update,ctr,avg_position,staleness_points,ctr_position_points
0,1,content_1b4ec72dafd4,6,STALE_PLUS_LOW_CTR,REFRESH,372,0.0,7.0,3,3
1,2,content_55a5b1c46474,6,STALE_PLUS_LOW_CTR,REFRESH,373,0.0,7.5,3,3
2,3,content_02b0d6e30129,5,STALE_PLUS_LOW_CTR,REFRESH,313,0.0,6.9,2,3
3,4,content_06e19c6486b0,5,STALE_PLUS_LOW_CTR,REFRESH,334,0.0,5.0,2,3
4,5,content_07ce98c6085a,5,STALE_PLUS_LOW_CTR,REFRESH,304,0.0,5.3,2,3
5,6,content_0af51f20a3da,5,STALE_PLUS_LOW_CTR,REFRESH,183,0.0,8.4,2,3
6,7,content_107776820988,5,STALE_PLUS_LOW_CTR,REFRESH,211,0.0,5.2,2,3
7,8,content_10b9f5f766b4,5,STALE_PLUS_LOW_CTR,REFRESH,211,0.0,5.7,2,3
8,9,content_10bd2440e4ac,5,STALE_PLUS_LOW_CTR,REFRESH,183,0.0,7.0,2,3
9,10,content_164eee6bf9c1,5,STALE_PLUS_LOW_CTR,REFRESH,183,0.0,5.8,2,3


Score distribution:
action_score
0    15433
1     9014
2     1240
3     3369
4      895
5       47
6        2
Name: count, dtype: int64

Reason-code distribution:
reason_code
NO_PRIORITY_SIGNAL    15433
STALE_REFRESH          7311
LOW_CTR_POSITION       5222
STALE_PLUS_LOW_CTR     2034
Name: count, dtype: int64

Action distribution:
action
MONITOR    15433
REFRESH    14567
Name: count, dtype: int64


,rank,content_id,action_score,reason_code,action,days_since_last_update,ctr,avg_position,staleness_points,ctr_position_points
0,1,content_1b4ec72dafd4,6,STALE_PLUS_LOW_CTR,REFRESH,372,0.0,7.0,3,3
1,2,content_55a5b1c46474,6,STALE_PLUS_LOW_CTR,REFRESH,373,0.0,7.5,3,3
2,3,content_02b0d6e30129,5,STALE_PLUS_LOW_CTR,REFRESH,313,0.0,6.9,2,3
3,4,content_06e19c6486b0,5,STALE_PLUS_LOW_CTR,REFRESH,334,0.0,5.0,2,3
4,5,content_07ce98c6085a,5,STALE_PLUS_LOW_CTR,REFRESH,304,0.0,5.3,2,3
5,6,content_0af51f20a3da,5,STALE_PLUS_LOW_CTR,REFRESH,183,0.0,8.4,2,3
6,7,content_107776820988,5,STALE_PLUS_LOW_CTR,REFRESH,211,0.0,5.2,2,3
7,8,content_10b9f5f766b4,5,STALE_PLUS_LOW_CTR,REFRESH,211,0.0,5.7,2,3
8,9,content_10bd2440e4ac,5,STALE_PLUS_LOW_CTR,REFRESH,183,0.0,7.0,2,3
9,10,content_164eee6bf9c1,5,STALE_PLUS_LOW_CTR,REFRESH,183,0.0,5.8,2,3


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Top-20 review

The top-20 queue is dominated by pages where both baseline signals contribute: the pages are relatively stale and have low observed CTR while ranking in a potentially useful position.

Confidence is moderate rather than high because both signal audits were MIXED. In particular, the 365+ day staleness bucket contains only five observations, so extreme staleness should not be treated as strong evidence by itself.

The main failure mode is that a low CTR may reflect SERP features, search intent, query mix, or measurement effects rather than content quality. Similarly, an old page may be intentionally evergreen and not need a refresh.

The review below records the action, reason code, confidence note, and what could make each recommendation wrong.

In [9]:
# ==========================================
# SECTION 3 — TOP-20 REVIEW
# ==========================================

top20 = output.head(20).copy()


def confidence_note(row):
    if row["reason_code"] == "STALE_PLUS_LOW_CTR":
        return (
            "Moderate: both signals contribute, but both "
            "signal audits were MIXED."
        )
    elif row["reason_code"] == "STALE_REFRESH":
        return (
            "Low-moderate: staleness supports review, "
            "but age alone does not prove refresh value."
        )
    elif row["reason_code"] == "LOW_CTR_POSITION":
        return (
            "Low-moderate: low CTR at a useful position "
            "is directional, not causal."
        )
    else:
        return "Low: no positive priority signal."


def what_would_make_it_wrong(row):
    if row["reason_code"] == "STALE_PLUS_LOW_CTR":
        return (
            "Low CTR may reflect SERP features, search intent, "
            "query mix, or measurement effects; the content may "
            "also still be accurate."
        )
    elif row["reason_code"] == "STALE_REFRESH":
        return (
            "The page may be intentionally evergreen, so age "
            "alone may not justify a refresh."
        )
    elif row["reason_code"] == "LOW_CTR_POSITION":
        return (
            "Low CTR may be caused by SERP layout or search "
            "intent rather than weak content."
        )
    else:
        return (
            "The rule does not provide strong evidence for "
            "prioritizing this row."
        )


top20_review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "action_score",
    ]
].copy()

top20_review["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20_review["what_would_make_it_wrong"] = top20.apply(
    what_would_make_it_wrong,
    axis=1
)

display(top20_review)

,rank,content_id,action,reason_code,action_score,confidence_note,what_would_make_it_wrong
0,1,content_1b4ec72dafd4,REFRESH,STALE_PLUS_LOW_CTR,6,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
1,2,content_55a5b1c46474,REFRESH,STALE_PLUS_LOW_CTR,6,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
2,3,content_02b0d6e30129,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
3,4,content_06e19c6486b0,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
4,5,content_07ce98c6085a,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
5,6,content_0af51f20a3da,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
6,7,content_107776820988,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
7,8,content_10b9f5f766b4,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
8,9,content_10bd2440e4ac,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."
9,10,content_164eee6bf9c1,REFRESH,STALE_PLUS_LOW_CTR,5,"Moderate: both signals contribute, but both si...","Low CTR may reflect SERP features, search inte..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Weak picks + leakage check

### Weak picks

The top-ranked rows are not automatically correct recommendations. The main weak-pick risk is that the baseline gives substantial weight to low CTR and staleness even though both signal audits were MIXED.

The 365+ day bucket contains only five observations, so very old content has limited observed support in this dataset. Low CTR can also be influenced by SERP features, search intent, query mix, or measurement effects rather than content quality.

Therefore, the top queue should be treated as a review shortlist rather than a guaranteed refresh list.

### Leakage check

The score uses only `days_since_last_update`, `ctr`, and `avg_position`.

I did not use product flags, client identifiers, private queries, labels, or future-window information. `trend_pct` and `trend_direction` are also excluded from the scoring rule.

The generated CSV is a reproducible ranked decision-support queue and is not treated as ground truth.

In [11]:
# ==========================================
# SECTION 4 — WEAK PICKS + LEAKAGE CHECK
# ==========================================

# Features used by the baseline
score_features = {
    "days_since_last_update",
    "ctr",
    "avg_position"
}

# Fields explicitly excluded from scoring
excluded_fields = {
    "client_id",
    "trend_pct",
    "trend_direction"
}

# Verify required scoring features exist
missing_score_features = [
    c for c in score_features
    if c not in df.columns
]

assert not missing_score_features, (
    f"Missing scoring features: {missing_score_features}"
)

# Verify excluded fields are not scoring features
assert score_features.isdisjoint(excluded_fields)

# Verify queue integrity
assert len(output) == len(df)
assert output["rank"].is_monotonic_increasing
assert output["action_score"].notna().all()
assert output["reason_code"].notna().all()
assert output["action"].notna().all()

print("==========================================")
print("LEAKAGE / INTEGRITY CHECK")
print("==========================================")

print("PASS — scoring features:")
print(sorted(score_features))

print("\nPASS — explicitly excluded fields:")
print(sorted(excluded_fields))

print("\nRows ranked:", len(output))

print("\nReason-code distribution:")
display(output["reason_code"].value_counts())

print("\nAction distribution:")
display(output["action"].value_counts())

print("\nPASS — no product flags or future-window fields were used in the score.")

LEAKAGE / INTEGRITY CHECK
PASS — scoring features:
['avg_position', 'ctr', 'days_since_last_update']

PASS — explicitly excluded fields:
['client_id', 'trend_direction', 'trend_pct']

Rows ranked: 30000

Reason-code distribution:


,count
reason_code,
NO_PRIORITY_SIGNAL,15433
STALE_REFRESH,7311
LOW_CTR_POSITION,5222
STALE_PLUS_LOW_CTR,2034



Action distribution:


,count
action,
MONITOR,15433
REFRESH,14567



PASS — no product flags or future-window fields were used in the score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.